# Lab 1 — Working Environment and a First End-to-End Pipeline
**Machine Learning I · PEU-CD 2026 · ENEI**

Companion to `tutorial.pdf`. Cells marked `# TODO` are yours. Run top to bottom.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score

rng = np.random.default_rng(155)
np.set_printoptions(precision=4, suppress=True)

## 1. numpy warm-up
Shapes, `@`, broadcasting, `solve`. Nothing to submit here — just run and read.

In [ ]:
A = rng.normal(size=(4, 3))
print("A.shape        =", A.shape)
print("(A.T @ A).shape=", (A.T @ A).shape)
print("column means   =", A.mean(axis=0))
print("centred means  =", (A - A.mean(axis=0)).mean(axis=0))   # broadcasting
b = rng.normal(size=3)
M = A.T @ A
print("solve vs inv agree:", np.allclose(np.linalg.solve(M, b), np.linalg.inv(M) @ b))

## 2. Lecture 1, recomputed
### Task 1 — normal equations by hand
Expected: $\hat\beta = (1.49, 0.75)$.

In [ ]:
x = np.array([1, 2, 3, 4, 5.0])
y = np.array([2.2, 2.8, 4.5, 3.7, 5.5])
X = np.column_stack([np.ones_like(x), x])          # design matrix, intercept first

# TODO: form X^T X and X^T y, solve the normal equations with np.linalg.solve
XtX = ...
Xty = ...
beta_hat = ...
print("beta_hat =", beta_hat)
assert np.allclose(beta_hat, [1.49, 0.75])

### Task 2 — four properties of the hat matrix

In [ ]:
# TODO: build H without calling np.linalg.inv, then check the four properties with np.allclose
H = ...
r = y - H @ y
print("symmetric  :", ...)
print("idempotent :", ...)
print("trace      :", ...)
print("X^T r = 0  :", ...)
print("eigenvalues:", np.sort(np.linalg.eigvalsh(H))[::-1].round(6))
# Why are the eigenvalues what they are? Write one sentence:
#

### Task 3 — gradient descent: implement, then break it

In [ ]:
def gradient_descent(X, y, eta, iters, beta0=None):
    N, d = X.shape
    beta = np.zeros(d) if beta0 is None else beta0.copy()
    losses = []
    for _ in range(iters):
        # TODO: gradient of L(beta) = (1/2N) ||y - X beta||^2, then the update
        grad = ...
        beta = ...
        losses.append(0.5 * np.mean((y - X @ beta) ** 2))
    return beta, np.array(losses)

beta_gd, L = gradient_descent(X, y, eta=0.05, iters=2000)
print("GD  :", beta_gd, "  closed form:", beta_hat)
assert np.allclose(beta_gd, beta_hat, atol=1e-3)

# TODO: lambda_max of X^T X / N, critical step 2/lambda_max, then plot the loss for eta just below and just above it
lam_max = ...
eta_crit = ...

# TODO: standardize x, rebuild the design matrix, and report the new lambda_max and iterations to converge


## 3. A real pipeline
`load_diabetes`: 442 patients, 10 standardized covariates, disease progression as target.

In [ ]:
data = load_diabetes()
X_all, y_all = data.data, data.target
print(X_all.shape, y_all.shape)
X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.2, random_state=155)
print("train:", X_tr.shape, " test (sealed):", X_te.shape)

### Task 4 — two numbers to beat

In [ ]:
# TODO: baseline MSE (predict the training mean), then LinearRegression train MSE and R^2
baseline_mse = ...
model = LinearRegression().fit(X_tr, y_tr)
train_mse = ...
print(f"baseline MSE = {baseline_mse:.1f}")
print(f"linear   MSE = {train_mse:.1f}")

# TODO: solve the normal equations on (X_tr with intercept column) and confirm model.coef_ matches


### Task 5 — K-fold cross-validation, by hand

In [ ]:
def kfold_mse(X, y, K, make_model, seed=155):
    kf = KFold(n_splits=K, shuffle=True, random_state=seed)
    scores = []
    for tr, va in kf.split(X):
        # TODO: fit on the K-1 folds, evaluate MSE on the held-out fold
        ...
    return np.array(scores)

mine = kfold_mse(X_tr, y_tr, 5, LinearRegression)
theirs = -cross_val_score(LinearRegression(), X_tr, y_tr, cv=KFold(5, shuffle=True, random_state=155),
                          scoring="neg_mean_squared_error")
print("mine  :", mine.round(1)); print("theirs:", theirs.round(1))
assert np.allclose(mine, theirs)

### Task 6 — polynomial degree: the bias–variance picture, for real

In [ ]:
degrees = [1, 2, 3]
tr_err, cv_err = [], []
for d in degrees:
    make = lambda d=d: make_pipeline(PolynomialFeatures(d, include_bias=False), StandardScaler(), LinearRegression())
    # TODO: training MSE of the fitted pipeline, and mean 5-fold CV MSE via kfold_mse
    ...
# TODO: plot both curves against degree, choose best_degree by CV, and explain each curve in one sentence
best_degree = ...


### Task 7 — learning curves

In [ ]:
X_a, X_v, y_a, y_v = train_test_split(X_tr, y_tr, test_size=0.2, random_state=155)
sizes = [20, 40, 80, 160, len(y_a)]
# TODO: for each n, fit on the first n rows of (X_a, y_a); record train MSE on those rows and MSE on (X_v, y_v)
# TODO: plot both against n (log x-axis)


### The one number
Open the test set once.

In [ ]:
# TODO: fit the chosen pipeline on the full training set and report MSE on X_te. Then stop.


## 4. Exercises — see `tutorial.pdf`, Section 6
Add your cells below.